# Gastrula-to-Pup multimodal training with RegularizedMultimodalVI

This notebook trains a **3-modality model** (total RNA + spliced + unspliced) on the
Qiu et al. 2024 mouse prenatal development dataset using `RegularizedMultimodalVI`.

**Key features**:
- **Purpose-driven covariate keys** designed for sci-RNA-seq3 batch structure
- **3-level ambient RNA modelling**: RT-step (per-embryo) + protease-step (per-PCR-well)
- **Step-based training** with checkpointing (~25 epochs over 12M cells)
- **Decoder attribution analysis**: which latent dimensions does each modality's decoder use?

**Prerequisites**: Run `01_data_download_and_preparation.ipynb` first to produce
the unfiltered checkpoint at `gastrula_to_pup_unfiltered.h5ad`.

In [ ]:
import gc
import os
import shutil
import time

import matplotlib.pyplot as plt
import mudata as mu
import numpy as np
import pandas as pd
import scanpy as sc
import torch
from matplotlib import rcParams

import regularizedvi
import scvi

scvi.settings.progress_bar_style = "tqdm"
torch.set_float32_matmul_precision("high")
rcParams["pdf.fonttype"] = 42

DATA_DIR = "/nfs/team205/vk7/sanger_projects/large_data/gastrula_to_pup"
results_folder = "results/gastrula_to_pup/"
os.makedirs(results_folder, exist_ok=True)

## 1. Load MuData

Load the gene-filtered, QC-annotated MuData produced by Notebook 1.
Each modality (rna, spliced, unspliced) has already been subset to informative genes.

In [ ]:
h5mu_path = os.path.join(DATA_DIR, "gastrula_to_pup_filtered.h5mu")
mdata = mu.read_h5mu(h5mu_path)
print(mdata)
for mod_name in mdata.mod:
    print(f"  {mod_name}: {mdata[mod_name].shape}, dtype={mdata[mod_name].X.dtype}")
print(f"\nObs columns: {list(mdata['rna'].obs.columns)}")

## 2. Cell filtering

sci-RNA-seq3 has lower UMI counts per cell (~2700 median) than 10x (~5000+),
so we use more lenient thresholds. Adjust based on QC plots from Notebook 1.

In [ ]:
n_before = mdata["rna"].n_obs

# QC thresholds from any modality (obs is shared across all modalities)
obs = mdata["rna"].obs
cell_mask = (
    (obs["n_genes_by_counts"] > 200)  # minimum gene complexity
    & (obs["total_counts"] > 500)  # minimum UMI (lenient for sci-RNA-seq3)
    & (obs["total_counts"] < 50000)  # upper UMI bound
    & (obs["mt_frac"] < 0.10)  # mitochondrial fraction
    & (obs["doublet_score"] < 0.20)  # doublet filter
)

# NOTE: Thresholds may need adjustment based on QC plots from Notebook 1.

# Filter all modalities
mdata = mu.MuData({mod_name: mdata[mod_name][cell_mask.values].copy() for mod_name in mdata.mod})
gc.collect()
print(f"Filtered {n_before} -> {mdata['rna'].n_obs} cells ({n_before - mdata['rna'].n_obs} removed)")
for mod_name in mdata.mod:
    print(f"  {mod_name}: {mdata[mod_name].shape}")

## 3. I/O benchmark: in-memory vs disk-backed training

MuData supports **backed mode** (`mu.read_h5mu(path, backed="r")`) where count
matrices stay on disk and minibatches are loaded via h5py during training. This can
dramatically reduce RAM usage at the cost of slower I/O.

We benchmark three options:
1. **In-memory** (default) — fastest, but requires all data in RAM
2. **Backed from network lustre** — minimal RAM, but network I/O overhead
3. **Backed from local `/tmp`** — copy to local SSD first, then back from there

In [ ]:
# Save filtered MuData for backed mode and future use
h5mu_filt = os.path.join(DATA_DIR, "gastrula_to_pup_filtered_qc.h5mu")
mdata.write_h5mu(h5mu_filt)
print(f"Saved filtered MuData: {h5mu_filt} ({os.path.getsize(h5mu_filt) / 1e9:.1f} GB)")

# Benchmark minibatch reads (100 random batches of 1024 cells, all modalities)
n_cells = mdata["rna"].n_obs
batch_idx = np.sort(np.random.choice(n_cells, 1024, replace=False))
N_READS = 100

# Option A: In-memory
t0 = time.time()
for _ in range(N_READS):
    for mod_name in mdata.mod:
        _ = mdata[mod_name].X[batch_idx]
t_mem = time.time() - t0

# Option B: Backed from lustre
mdata_lustre = mu.read_h5mu(h5mu_filt, backed="r")
t0 = time.time()
for _ in range(N_READS):
    for mod_name in mdata_lustre.mod:
        _ = mdata_lustre[mod_name].X[batch_idx]
t_lustre = time.time() - t0
mdata_lustre.file.close()
del mdata_lustre

# Option C: Backed from local /tmp
local_path = "/tmp/gastrula_to_pup_filtered_qc.h5mu"
shutil.copy2(h5mu_filt, local_path)
mdata_local = mu.read_h5mu(local_path, backed="r")
t0 = time.time()
for _ in range(N_READS):
    for mod_name in mdata_local.mod:
        _ = mdata_local[mod_name].X[batch_idx]
t_local = time.time() - t0
mdata_local.file.close()
del mdata_local

print(f"\n{'Option':<25} {'Time (s)':>10} {'Relative':>10}")
print("-" * 47)
print(f"{'In-memory':<25} {t_mem:10.2f} {'1.0x':>10}")
print(f"{'Backed (lustre)':<25} {t_lustre:10.2f} {f'{t_lustre / t_mem:.1f}x':>10}")
print(f"{'Backed (/tmp local)':<25} {t_local:10.2f} {f'{t_local / t_mem:.1f}x':>10}")

# Estimate full training time impact (25 epochs × ~12K steps/epoch)
total_reads = 25 * 12000
print("\nEstimated total training I/O overhead:")
print(f"  In-memory: {total_reads * t_mem / N_READS / 3600:.1f} hours")
print(f"  Backed (lustre): {total_reads * t_lustre / N_READS / 3600:.1f} hours")
print(f"  Backed (/tmp): {total_reads * t_local / N_READS / 3600:.1f} hours")

In [ ]:
# Choose training data source based on benchmark results
# Uncomment ONE of the following options:

# Option A: In-memory (default — fastest, requires sufficient RAM)
# mdata is already in memory from cell filtering above

# Option B: Backed from /tmp (lower RAM, moderate speed)
# mdata = mu.read_h5mu("/tmp/gastrula_to_pup_filtered_qc.h5mu", backed="r")

# Option C: Backed from lustre (lowest RAM, slowest)
# mdata = mu.read_h5mu(h5mu_filt, backed="r")

print(f"Training with: {'in-memory' if not mdata.isbacked else 'backed'} MuData")
print(f"  isbacked: {mdata.isbacked}")
for mod_name in mdata.mod:
    print(f"  {mod_name}: {mdata[mod_name].shape}")

## 4. Model setup — covariate design for sci-RNA-seq3

sci-RNA-seq3 uses **3-level combinatorial indexing**, each introducing distinct
technical effects. We assign each effect to the appropriate covariate key:

| Key | Value | Rationale |
|-----|-------|-----------|
| `ambient_covariate_keys` | `["embryo_id", "pcr_well"]` | **RT-step ambient** (dominant, per-embryo tissue lysis) + **protease-step ambient** (per-PCR-well, absorbs experiment drift) |
| `categorical_covariate_keys` | `["experimental_batch"]` | Multiplicative reagent/sequencing biases (15 levels, low overcorrection risk) |
| `modality_scaling_covariate_keys` | `None` → inherits `["experimental_batch"]` | Region factors mirror categorical by default |
| `dispersion_key` | `"embryo_experiment"` | Composite of embryo_id × experimental_batch |
| `library_size_key` | `"pcr_well"` | Sequencing budget allocated per PCR well |

**Why not `embryo_id` in `categorical_covariate_keys`?**
Embryos span different developmental stages — using embryo_id as a multiplicative
covariate would absorb real biological variation (cell type composition changes across
development). The ambient model (additive) is the right place for embryo_id.

In [ ]:
regularizedvi.RegularizedMultimodalVI.setup_mudata(
    mdata,
    ambient_covariate_keys=["embryo_id", "pcr_well"],
    categorical_covariate_keys=["experimental_batch"],
    modality_scaling_covariate_keys=None,  # inherits from categorical
    dispersion_key="embryo_experiment",
    library_size_key="pcr_well",
)

## 5. Model initialisation

Large architecture for a 12M-cell dataset:
- RNA: 2048 hidden, 512 latent (dominant modality)
- Spliced/Unspliced: 512 hidden, 128 latent each
- Total latent dim = 512 + 128 + 128 = 768

All 3 modalities get additive background correction and region factors (symmetric design).

In [ ]:
# === CONFIGURABLE HYPERPARAMETERS ===
N_HIDDEN = {"rna": 2048, "spliced": 512, "unspliced": 512}
N_LATENT = {"rna": 512, "spliced": 128, "unspliced": 128}
N_LAYERS = 1
BATCH_SIZE = 1024
MAX_EPOCHS = 25  # ~300k steps with 12M cells and batch_size=1024
CHECKPOINT_EVERY_N_EPOCHS = 1

model = regularizedvi.RegularizedMultimodalVI(
    mdata,
    n_hidden=N_HIDDEN,
    n_latent=N_LATENT,
    n_layers=N_LAYERS,
    latent_mode="concatenation",
    additive_background_modalities=["rna", "spliced", "unspliced"],
    region_factors_modalities=["rna", "spliced", "unspliced"],
    dispersion="gene-batch",
    regularise_dispersion=True,
)

print(model)
print(f"\nTotal latent dim: {model.module.total_latent_dim}")
for name in model.module.modality_names:
    print(f"  Z_{name} dim: {model.module.n_latent_dict[name]}")
print(f"\nAdditive bg modalities: {model.module.additive_background_modalities}")
print(f"Region factors modalities: {model.module.region_factors_modalities}")

## 6. Training with checkpointing

With ~12M cells and `batch_size=1024`, one epoch ≈ 12K steps.
`max_epochs=25` gives ~300K total steps. Early stopping monitors validation ELBO.

Checkpoints are saved every epoch to allow resuming from any point.

In [ ]:
from scvi.train import SaveCheckpoint

checkpoint_cb = SaveCheckpoint(
    dirpath=os.path.join(results_folder, "checkpoints"),
    every_n_epochs=CHECKPOINT_EVERY_N_EPOCHS,
    save_top_k=-1,  # keep all checkpoints
    filename="epoch-{epoch}",
)

model.train(
    train_size=0.9,
    max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    early_stopping=True,
    early_stopping_patience=5,
    early_stopping_monitor="elbo_validation",
    check_val_every_n_epoch=1,
    enable_checkpointing=True,
    callbacks=[checkpoint_cb],
)

In [ ]:
# Training loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
skip = 2  # skip first epochs for better scale

axes[0].plot(model.history_["elbo_train"].iloc[skip:])
if "elbo_validation" in model.history_:
    axes[0].plot(model.history_["elbo_validation"].iloc[skip:], label="validation")
    axes[0].legend()
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("ELBO")
axes[0].set_title("ELBO")

axes[1].plot(model.history_["reconstruction_loss_train"].iloc[skip:])
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Reconstruction loss")
axes[1].set_title("Reconstruction loss")

plt.tight_layout()
plt.show()

In [ ]:
# Per-modality training diagnostics
skip = 2
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Per-modality reconstruction loss
ax = axes[0]
for name in model.module.modality_names:
    key = f"recon_loss_{name}_train"
    if key in model.history_:
        ax.plot(model.history_[key].iloc[skip:].values, label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Recon loss")
ax.set_title("Per-modality recon loss")
ax.legend()

# Per-modality KL on Z
ax = axes[1]
for name in model.module.modality_names:
    key = f"kl_z_{name}_train"
    if key in model.history_:
        ax.plot(model.history_[key].iloc[skip:].values, label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("KL(Z)")
ax.set_title("Per-modality KL")
ax.legend()

# Per-modality Z variance
ax = axes[2]
for name in model.module.modality_names:
    key = f"z_var_{name}_train"
    if key in model.history_:
        ax.plot(model.history_[key].iloc[skip:].values, label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Z variance")
ax.set_title("Latent utilisation")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Save model
model.save(os.path.join(results_folder, "model"), overwrite=True)

## 7. Latent space extraction

With `latent_mode='concatenation'`, Z = [Z_rna; Z_spliced; Z_unspliced].
We extract the full latent representation and split it into modality-specific slices.

In [ ]:
latent = model.get_latent_representation()
print(f"Full latent shape: {latent.shape}")

# Split by modality (alphabetical order in modality_names)
z_parts = {}
offset = 0
for name in model.module.modality_names:
    n_lat = model.module.n_latent_dict[name]
    z_parts[name] = latent[:, offset : offset + n_lat]
    offset += n_lat
    print(f"  Z_{name}: [{offset - n_lat}, {offset}) -> shape {z_parts[name].shape}")

# Store in mdata for plotting (use first modality's obs)
ref_mod = model.module.modality_names[0]
ref_adata = mdata[ref_mod]
ref_adata.obsm["X_multiVI_joint"] = latent
for name in model.module.modality_names:
    ref_adata.obsm[f"X_multiVI_{name}"] = z_parts[name]

In [ ]:
# Compute UMAPs (joint + per-modality)
k = 50

# Joint UMAP
sc.pp.neighbors(ref_adata, use_rep="X_multiVI_joint", n_neighbors=k, key_added="joint")
sc.tl.umap(ref_adata, min_dist=0.4, spread=1.3, neighbors_key="joint")
ref_adata.obsm["X_umap_joint"] = ref_adata.obsm["X_umap"].copy()

# Per-modality UMAPs
for name in model.module.modality_names:
    sc.pp.neighbors(ref_adata, use_rep=f"X_multiVI_{name}", n_neighbors=k, key_added=name)
    sc.tl.umap(ref_adata, min_dist=0.4, spread=1.3, neighbors_key=name)
    ref_adata.obsm[f"X_umap_{name}"] = ref_adata.obsm["X_umap"].copy()

print("Computed 4 UMAP embeddings: joint + rna + spliced + unspliced")

In [ ]:
# Joint UMAP coloured by metadata
color = ["day", "embryo_id", "experimental_batch", "major_trajectory", "celltype_update"]
# Filter to columns that exist
color = [c for c in color if c in ref_adata.obs.columns]

rcParams["figure.figsize"] = 7, 7
ref_adata.obsm["X_umap"] = ref_adata.obsm["X_umap_joint"]

sc.pl.umap(
    ref_adata,
    color=color,
    ncols=1,
    size=0.5,
    title=[f"Joint Z - {c}" for c in color],
)

In [ ]:
# Side-by-side modality UMAPs
n_panels = len(model.module.modality_names) + 1
fig, axes = plt.subplots(1, n_panels, figsize=(7 * n_panels, 7))

umap_keys = [("X_umap_joint", "Joint Z")] + [(f"X_umap_{n}", f"{n} Z") for n in model.module.modality_names]

plot_color = "celltype_update" if "celltype_update" in ref_adata.obs.columns else "major_trajectory"

for ax, (umap_key, title) in zip(axes, umap_keys, strict=False):
    ref_adata.obsm["X_umap"] = ref_adata.obsm[umap_key]
    sc.pl.umap(
        ref_adata,
        color=plot_color,
        size=0.5,
        ax=ax,
        show=False,
        title=title,
        legend_loc="none",
    )

plt.tight_layout()
plt.show()

## 8. Decoder attribution analysis

Compute the **mean absolute Jacobian** `|d(px_rate)/d(z)|` per latent dimension for
each modality's decoder. This reveals which parts of the shared
Z = [Z_rna; Z_spliced; Z_unspliced] inform each modality.

**What to look for**:
- If each decoder primarily uses its own Z dims, the modalities learned independent representations.
- If spliced/unspliced decoders rely on Z_rna dims, they are **free-riding** on RNA-derived information.
- Total RNA is expected to dominate (most information), with spliced/unspliced providing
  velocity-relevant signal.

In [ ]:
attribution = model.get_modality_attribution(batch_size=256)

for name in model.module.modality_names:
    attr = attribution[name]["attribution"]
    print(f"{name} attribution: {attr.shape}, range [{attr.mean(0).min():.4f}, {attr.mean(0).max():.4f}]")

In [ ]:
# Attribution bar plots
n_mods = len(model.module.modality_names)
fig, axes = plt.subplots(1, n_mods, figsize=(7 * n_mods, 5))
if n_mods == 1:
    axes = [axes]

mod_colors = {"rna": "tab:blue", "spliced": "tab:green", "unspliced": "tab:red"}

for ax, name in zip(axes, model.module.modality_names, strict=False):
    attr = attribution[name]["attribution"]
    mean_attr = attr.mean(axis=0)

    # Colour by source modality
    colors = []
    for mod_name in model.module.modality_names:
        n = model.module.n_latent_dict[mod_name]
        c = mod_colors.get(mod_name, "tab:gray")
        colors.extend([c] * n)

    ax.bar(range(len(mean_attr)), mean_attr, color=colors, alpha=0.7)
    ax.set_xlabel("Latent dim")
    ax.set_ylabel("Mean |Jacobian|")
    ax.set_title(f"{name} decoder attribution")

    # Add modality boundaries
    offset = 0
    for mod_name in model.module.modality_names[:-1]:
        offset += model.module.n_latent_dict[mod_name]
        ax.axvline(offset - 0.5, color="red", linestyle="--", alpha=0.5)

    # Legend
    from matplotlib.patches import Patch

    legend_elements = [
        Patch(facecolor=mod_colors.get(mn, "tab:gray"), alpha=0.7, label=f"Z_{mn}")
        for mn in model.module.modality_names
    ]
    ax.legend(handles=legend_elements)

plt.tight_layout()
plt.show()

# Cross-modality attribution fractions
for name in model.module.modality_names:
    attr = attribution[name]["attribution"]
    total = attr.mean(axis=0).sum()
    offset = 0
    for mod_name in model.module.modality_names:
        n = model.module.n_latent_dict[mod_name]
        frac = attr.mean(axis=0)[offset : offset + n].sum() / total
        print(f"  {name} decoder: {frac:.1%} from Z_{mod_name}")
        offset += n
    print()

In [ ]:
# Attribution-weighted UMAPs
for name in model.module.modality_names:
    ref_adata.obsm[f"X_multiVI_attr_{name}"] = attribution[name]["weighted_z"]
    sc.pp.neighbors(
        ref_adata,
        use_rep=f"X_multiVI_attr_{name}",
        n_neighbors=k,
        key_added=f"attr_{name}",
    )
    sc.tl.umap(ref_adata, min_dist=0.4, spread=1.3, neighbors_key=f"attr_{name}")
    ref_adata.obsm[f"X_umap_attr_{name}"] = ref_adata.obsm["X_umap"].copy()

# Compare attribution UMAPs
n_panels = n_mods + 1
fig, axes = plt.subplots(1, n_panels, figsize=(7 * n_panels, 7))
umap_keys = [("X_umap_joint", "Joint Z")] + [(f"X_umap_attr_{n}", f"{n} attr Z") for n in model.module.modality_names]

for ax, (umap_key, title) in zip(axes, umap_keys, strict=False):
    ref_adata.obsm["X_umap"] = ref_adata.obsm[umap_key]
    sc.pl.umap(
        ref_adata,
        color=plot_color,
        size=0.5,
        ax=ax,
        show=False,
        title=title,
        legend_loc="none",
    )

plt.tight_layout()
plt.show()

## 9. Save outputs

In [ ]:
output_dir = os.path.join(results_folder, "model", "outputs")
os.makedirs(output_dir, exist_ok=True)

# Save latent representations
for key in ["X_multiVI_joint"] + [f"X_multiVI_{n}" for n in model.module.modality_names]:
    df = pd.DataFrame(ref_adata.obsm[key], index=ref_adata.obs_names)
    df.to_csv(os.path.join(output_dir, f"{key}.csv"))

# Save UMAPs
umap_keys_to_save = (
    ["X_umap_joint"]
    + [f"X_umap_{n}" for n in model.module.modality_names]
    + [f"X_umap_attr_{n}" for n in model.module.modality_names]
)
for umap_key in umap_keys_to_save:
    if umap_key in ref_adata.obsm:
        df = pd.DataFrame(
            ref_adata.obsm[umap_key],
            index=ref_adata.obs_names,
            columns=["UMAP1", "UMAP2"],
        )
        df.to_csv(os.path.join(output_dir, f"{umap_key}_k{k}.csv"))

# Save attribution
for name in model.module.modality_names:
    for attr_key in ["attribution", "weighted_z"]:
        df = pd.DataFrame(attribution[name][attr_key], index=ref_adata.obs_names)
        df.to_csv(os.path.join(output_dir, f"attribution_{name}_{attr_key}.csv"))

print(f"Outputs saved to {output_dir}")

## Summary

This notebook demonstrated **RegularizedMultimodalVI** with 3 modalities on the
Qiu et al. 2024 gastrula-to-pup dataset:

1. **Purpose-driven covariate keys** — properly assigned to sci-RNA-seq3 batch hierarchy
   (RT-step ambient, protease-step ambient, experiment-level multiplicative effects)
2. **3-modality architecture** — total RNA (512 latent), spliced (128), unspliced (128)
   with symmetric additive background and region factors
3. **Decoder attribution** — reveals which modalities each decoder relies on;
   total RNA expected to dominate, spliced/unspliced provide velocity-relevant signal
4. **4 UMAP views** — joint Z, RNA Z, spliced Z, unspliced Z reveal modality-specific
   and shared cell identity